# ISBP — Exp 1 & Exp 2 (Regular Instances, w ∈ {4,...,15})

**Exp 1**: Best PNE and POS (BRD + ZR @ alpha=1)
- BRD warm-starts ZR (if PNE found); ZR optimises to find the **best** (min total cost) PNE
- SO = social optimum (min total cost without equilibrium constraints)
- POS = best_PNE_cost / SO_cost (≥ 1, closer to 1 = less inefficiency)

**Exp 2**: Tightest Alpha Search (TAS) for non-PNE instances
- Uses `stop_at_first_pne=True` (only needs existence, not optimality)

> **Note**: Exp 1B (VEST cut off) is in a separate notebook `02b_run_isbp_exp1b.ipynb`
> to ensure independent execution environment (eliminates memory/ordering bias).

In [ ]:
from pathlib import Path
import json
import time
import csv
import math
import traceback

import numpy as np
import pandas as pd

from gipg.isbp.instance import generate_isbp_instance
from gipg.isbp.heuristics import (
    brd_random_restart, alpha_of_profile,
    is_regular_instance, run_all_heuristics,
)
from gipg.isbp.gzr import solve_gzr
from gipg.isbp.abs_search import solve_abs
from gipg.isbp.objectives import profile_costs
from gipg.isbp.social_optimum import solve_social_optimum, compute_pos

OUTDIR = Path('outputs')
OUTDIR.mkdir(exist_ok=True)
print('Imports OK.')

## 1. Instance Generation

In [ ]:
def enumerate_tuples_regular(n_range, m_range, w_range):
    """
    Enumerate (n, m, w, u) using:
      - n in n_range
      - m in {n+m_range[0], ..., n+m_range[1]-1}
      - w in w_range
      - u candidates from floor(nw/m) with divisibility rule
    Keep only tuples satisfying feasibility: m*u - n*w >= 0.
    """
    tuples = []
    for n in n_range:
        for m in range(n + m_range[0], n + m_range[1]):
            if m <= 0:
                continue
            for w in w_range:
                nw = n * w
                q, r = divmod(nw, m)

                if r == 0:
                    u_candidates = [q, q+1, q+2, q+3]
                else:
                    u_candidates = [q+1, q+2, q+3, q+4]

                for u in u_candidates:
                    if (m*u - nw >= 0) and (m <= math.ceil(nw/u)):
                        tuples.append((n, m, w, u))
    return tuples

n_range = range(4, 11)
m_range = [-2, 4]     # m = n-2,...,n+3
w_range = range(4, 16) # w = 4,...,15

tuples = enumerate_tuples_regular(n_range, m_range, w_range)
print(f'#tuples kept (feasible by m*u - n*w >= 0): {len(tuples)}')
print('first 10:', tuples[:10])

In [ ]:
SEEDS = [0]  # e.g., [0,1,2,3,4]

# Controls
TOTAL_TIME_LIMIT = 600.0   # Total budget for BRD + ZR combined
SO_TIME_LIMIT = 600.0      # 10 minutes for social optimum
TAS_TIME_LIMIT = 600.0     # per ZR call in TAS
TAS_EPS = 1e-3

# Output CSV paths
exp1_path = OUTDIR / 'isbp_exp1_brd_zr_pos.csv'
exp2_path = OUTDIR / 'isbp_exp2_tas.csv'
combined_path = OUTDIR / 'isbp_results_all.csv'
err_path = OUTDIR / 'isbp_errors.csv'

# CSV append helper
def append_row(path: Path, header, rowdict):
    new_file = not path.exists()
    with path.open('a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=header)
        if new_file:
            w.writeheader()
        w.writerow({k: rowdict.get(k, '') for k in header})

print(f'Tuples: {len(tuples)}, Seeds: {len(SEEDS)}, Total runs: {len(tuples)*len(SEEDS)}')

## 2. Experiment 1: Best PNE and POS (BRD + ZR @ alpha=1)

For each instance:
1. **BRD** warm-starts **ZR @ alpha=1** (if BRD found a PNE)
2. **ZR** with `stop_at_first_pne=False` — optimises to MIP gap=0, finding the **best** PNE (min cost)
3. **Social Optimum** computed separately
4. **POS** = best_PNE_cost / SO_cost

In [ ]:
exp1_header = [
    'seed', 'n', 'm', 'w', 'u', 'regular',
    'brd_found_pne', 'brd_alpha', 'brd_time',
    'zr_status', 'zr_gurobi_status', 'zr_mip_gap', 'zr_obj_bound', 'zr_runtime', 'zr_time',
    'zr_eis_added_total', 'zr_eis_added_symmetric', 'zr_eis_added_core',
    'zr_first_pne_time',
    'best_pne_cost',
    'so_status', 'so_cost', 'so_time',
    'pos',
]

err_header = ['seed', 'n', 'm', 'w', 'u', 'stage', 'error', 'traceback']

results_exp1 = []

total = 0
for (n, m, w, u) in tuples:
    for seed in SEEDS:
        total += 1
        regular = True
        tag = f'n{n}_m{m}_w{w}_u{u}_s{seed}'

        row = {
            'seed': seed, 'n': n, 'm': m, 'w': w, 'u': u,
            'regular': regular,
        }

        try:
            inst = generate_isbp_instance(
                n_players=n, m_bins=m,
                c_j_range=w, u_j_range=u, w_i_range=w,
                seed=seed,
            )

            # --- Phase 1: BRD ---
            x_brd, brd_pne, brd_time = brd_random_restart(
                inst, max_init=3, max_round=15,
                seed=seed,
            )
            brd_alpha = alpha_of_profile(inst, x_brd) if x_brd else float('inf')
            row['brd_found_pne'] = brd_pne
            row['brd_alpha'] = brd_alpha
            row['brd_time'] = brd_time

            # --- Phase 2: ZR @ alpha=1, warm-started with BRD profile ---
            zr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)
            warm = x_brd if brd_pne else None

            zr = solve_gzr(
                inst,
                alpha=1.0,
                time_limit=zr_time_limit,
                log_to_console=0,
                stop_at_first_pne=False,
                warm_start=warm,
            )

            row['zr_status'] = zr.get('status')
            row['zr_gurobi_status'] = zr.get('gurobi_status')
            row['zr_mip_gap'] = zr.get('mip_gap')
            row['zr_obj_bound'] = zr.get('obj_bound')
            row['zr_runtime'] = zr.get('runtime')
            row['zr_time'] = brd_time + float(zr.get('runtime', 0))  # BRD + ZR combined
            row['zr_eis_added_total'] = zr.get('eis_added_total', 0)
            row['zr_eis_added_symmetric'] = zr.get('eis_added_symmetric', 0)
            row['zr_eis_added_core'] = zr.get('eis_added_core', 0)
            row['zr_first_pne_time'] = (brd_time + zr['first_pne_time']
                                                    if zr.get('first_pne_time') is not None else None)

            # Best PNE cost
            if zr.get('x_profile') is not None:
                _, total_cost = profile_costs(
                    players=inst.players, bins=inst.bins,
                    costs=inst.costs, x_profile=zr['x_profile'],
                )
                row['best_pne_cost'] = total_cost
            elif brd_pne and x_brd is not None:
                _, total_cost = profile_costs(
                    players=inst.players, bins=inst.bins,
                    costs=inst.costs, x_profile=x_brd,
                )
                row['best_pne_cost'] = total_cost
            else:
                row['best_pne_cost'] = None

            # --- Phase 3: Social Optimum ---
            so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
            row['so_status'] = so_res.status
            row['so_cost'] = so_res.opt_cost
            row['so_time'] = so_res.runtime

            # POS = best_PNE_cost / SO_cost (minimization: POS >= 1)
            if row.get('best_pne_cost') is not None and so_res.opt_cost is not None and so_res.opt_cost > 0:
                row['pos'] = compute_pos(so_res.opt_cost, row['best_pne_cost'])
            else:
                row['pos'] = None

            results_exp1.append(row)
            append_row(exp1_path, exp1_header, row)

            # Progress
            if total % 50 == 0 or total == len(tuples) * len(SEEDS):
                pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
                zr_str = f'{row["zr_status"]}'
                zr_t = f'{row["zr_time"]:.1f}s' if row.get('zr_time') is not None else '?s'
                pos_str = f'POS={row["pos"]:.3f}' if row.get('pos') is not None else 'POS=N/A'
                fpne_str = f'1stPNE={row["zr_first_pne_time"]:.1f}s' if row.get('zr_first_pne_time') is not None else '1stPNE=N/A'
                print(f'[{total}] {tag}: {pne_str} | ZR {zr_str} ({zr_t}) | {pos_str} | {fpne_str}')

        except Exception as e:
            append_row(err_path, err_header, {
                'seed': seed, 'n': n, 'm': m, 'w': w, 'u': u,
                'stage': 'EXP1', 'error': repr(e),
                'traceback': traceback.format_exc(),
            })
            print(f'  !! EXP1 failed ({tag}): {e}')
            continue

df_exp1 = pd.DataFrame(results_exp1)
print(f'\nExp1 complete: {len(df_exp1)} rows saved to {exp1_path}')


In [ ]:
# Experiment 1 Summary
print('=== Experiment 1 Summary ===')

n_total = len(df_exp1)
n_brd_pne = df_exp1['brd_found_pne'].sum()
n_zr_found = (df_exp1['zr_status'] == 'OPTIMAL').sum()
n_zr_inf = (df_exp1['zr_status'] == 'INFEASIBLE').sum()
n_zr_tl = (df_exp1['zr_status'] == 'TIME_LIMIT').sum()
n_pos = df_exp1['pos'].notna().sum()

print(f'Total instances: {n_total}')
print(f'BRD found PNE: {n_brd_pne} ({n_brd_pne/n_total:.1%})')
print(f'ZR OPTIMAL: {n_zr_found} ({n_zr_found/n_total:.1%})')
print(f'ZR INFEASIBLE: {n_zr_inf} ({n_zr_inf/n_total:.1%})')
print(f'ZR TIME_LIMIT: {n_zr_tl} ({n_zr_tl/n_total:.1%})')
print(f'POS computed: {n_pos} ({n_pos/n_total:.1%})')

# By (n, m)
print('\n--- By (n, m) ---')
summary = df_exp1.groupby(['n', 'm']).agg(
    count=('seed', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    zr_optimal_rate=('zr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_zr_time=('zr_time', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary)

## 3. Experiment 2: TAS for Non-PNE Instances

Only instances where Exp 1 ZR reported **INFEASIBLE** or **TIME_LIMIT** (and BRD found no PNE).
Uses binary search with verified/unverified bounds.

In [ ]:
# ── Load Exp1 results from CSV (skip re-running Exp1) ──────────────
# Run this cell instead of cells 6-7 if you already have results.

df_exp1 = pd.read_csv(exp1_path)
for col in ['n', 'm', 'w', 'u', 'seed']:
    df_exp1[col] = df_exp1[col].astype(int)
df_exp1['brd_found_pne'] = df_exp1['brd_found_pne'].astype(bool)
results_exp1 = df_exp1.to_dict(orient='records')
print(f'Loaded Exp1 from {exp1_path}: {len(df_exp1)} rows')
print(f'ZR status counts:\n{df_exp1["zr_status"].value_counts().to_string()}')

In [ ]:
exp2_header = [
    'seed', 'n', 'm', 'w', 'u', 'regular',
    'alpha_init', 'alpha_star',
    'alpha_ub_verified', 'alpha_lb_unverified', 'alpha_lb_verified',
    'trivial_from_heuristic', 'verified_through_binary_search',
    'n_bisect_iters',
    'tas_time_total',
]

# Gather failed instances from Exp1
failed_rows = [
    r for r in results_exp1
    if r.get('zr_status') in ('INFEASIBLE', 'TIME_LIMIT', 'UNKNOWN')
    and not r.get('brd_found_pne', False)
]
print(f'Exp2 candidates: {len(failed_rows)} instances')

results_exp2 = []

for count, exp1_row in enumerate(failed_rows):
    n = exp1_row['n']
    m = exp1_row['m']
    w = exp1_row['w']
    u = exp1_row['u']
    seed = exp1_row['seed']
    tag = f'n{n}_m{m}_w{w}_u{u}_s{seed}'

    try:
        inst = generate_isbp_instance(
            n_players=n, m_bins=m,
            c_j_range=w, u_j_range=u, w_i_range=w,
            seed=seed,
        )

        tas = solve_abs(
            inst,
            eps=TAS_EPS,
            time_limit_per_call=TAS_TIME_LIMIT,
            regular=True,
            log_to_console=0,
        )

        row2 = {
            'seed': seed, 'n': n, 'm': m, 'w': w, 'u': u,
            'regular': True,
            'alpha_init': tas.get('alpha_init'),
            'alpha_star': tas.get('alpha_star'),
            'alpha_ub_verified': tas.get('alpha_ub_verified'),
            'alpha_lb_unverified': tas.get('alpha_lb_unverified'),
            'alpha_lb_verified': tas.get('alpha_lb_verified'),
            'trivial_from_heuristic': bool(tas.get('trivial_from_heuristic')),
            'verified_through_binary_search': bool(tas.get('verified_through_binary_search')),
            'n_bisect_iters': int(tas.get('n_bisect_iters', 0)),
            'tas_time_total': float(tas.get('time_total', 0.0)),
        }
        results_exp2.append(row2)
        append_row(exp2_path, exp2_header, row2)

        print(f'[{count+1}/{len(failed_rows)}] {tag}: '
              f'alpha*={row2["alpha_star"]:.4f} '
              f'(lb_v={row2["alpha_lb_verified"]}, lb_u={row2["alpha_lb_unverified"]}) '
              f'{row2["n_bisect_iters"]} iters, {row2["tas_time_total"]:.1f}s')

    except Exception as e:
        append_row(err_path, err_header, {
            'seed': seed, 'n': n, 'm': m, 'w': w, 'u': u,
            'stage': 'EXP2', 'error': repr(e),
            'traceback': traceback.format_exc(),
        })
        print(f'  !! EXP2 failed ({tag}): {e}')
        continue

df_exp2 = pd.DataFrame(results_exp2)
if len(df_exp2) > 0:
    print(f'\nExp2 complete: {len(df_exp2)} rows saved to {exp2_path}')
else:
    print('\nNo instances needed TAS (all solved in Exp1).')

In [ ]:
# Experiment 2 Summary
print('=== Experiment 2: TAS Summary ===')
if len(df_exp2) > 0:
    print(f'Instances in TAS: {len(df_exp2)}')
    print(f'Mean alpha*: {df_exp2["alpha_star"].mean():.4f}')
    print(f'Max  alpha*: {df_exp2["alpha_star"].max():.4f}')
    n_verified_lb = df_exp2['alpha_lb_verified'].notna().sum()
    print(f'Verified lower bound: {n_verified_lb} ({n_verified_lb/len(df_exp2):.1%})')
    print(f'Mean bisection iters: {df_exp2["n_bisect_iters"].mean():.1f}')

    print('\n--- By (n, m) ---')
    summary2 = df_exp2.groupby(['n', 'm']).agg(
        count=('seed', 'count'),
        avg_alpha_star=('alpha_star', 'mean'),
        max_alpha_star=('alpha_star', 'max'),
        avg_iters=('n_bisect_iters', 'mean'),
    ).round(4)
    print(summary2)
else:
    print('No instances required TAS.')

## 4. Combined Results

In [ ]:
# Merge Exp1 and Exp2
df_exp1['tag'] = df_exp1.apply(
    lambda r: f"n{int(r['n'])}_m{int(r['m'])}_w{int(r['w'])}_u{int(r['u'])}_s{int(r['seed'])}",
    axis=1,
)

if len(df_exp2) > 0:
    df_exp2['tag'] = df_exp2.apply(
        lambda r: f"n{int(r['n'])}_m{int(r['m'])}_w{int(r['w'])}_u{int(r['u'])}_s{int(r['seed'])}",
        axis=1,
    )
    exp2_merge_cols = ['tag', 'alpha_init', 'alpha_star', 'alpha_lb_verified',
                       'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters',
                       'tas_time_total']
    df_all = pd.merge(df_exp1, df_exp2[exp2_merge_cols], on='tag', how='left')
else:
    df_all = df_exp1.copy()
    for col in ['alpha_init', 'alpha_star', 'alpha_lb_verified',
                'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters',
                'tas_time_total']:
        df_all[col] = np.nan

# For instances solved in Exp1, alpha_star = 1.0
mask_solved = (df_all['zr_status'] == 'FOUND') | (df_all['brd_found_pne'] == True)
df_all.loc[mask_solved & df_all['alpha_star'].isna(), 'alpha_star'] = 1.0

df_all.to_csv(combined_path, index=False)
print(f'Combined results: {len(df_all)} rows -> {combined_path}')
print(f'Columns: {list(df_all.columns)}')

# Final summary
print('\n=== Final Summary ===')
for n_val in sorted(df_all['n'].unique()):
    sub = df_all[df_all['n'] == n_val]
    n_pne = ((sub['zr_status'] == 'FOUND') | (sub['brd_found_pne'] == True)).sum()
    pos_ok = sub['pos'].notna()
    pos_str = f'POS mean={sub.loc[pos_ok, "pos"].mean():.3f}' if pos_ok.any() else 'no POS'
    tas_sub = sub[sub['n_bisect_iters'].notna() & (sub['n_bisect_iters'] > 0)]
    tas_str = f'TAS: {len(tas_sub)} inst' if len(tas_sub) > 0 else 'no TAS'
    print(f'  n={n_val}: {len(sub)} inst, PNE={n_pne}, {pos_str}, {tas_str}')